# Forgetful causal masking · SimpleStories 5M

This notebook compares clean causal training, uniform Forgetful Causal Masking (FCM), and uniform FCM with a linearly decaying masking rate (MRD). Selected input positions are hidden as attention keys during training; next-token targets remain unchanged. Evaluation and generation use clean causal attention.

**Colab setup:** connect to a Colab GPU kernel, make the `harness` folder available at `/content/harness`, and authorize the Google Drive mount at startup. Run folders are written directly under `MyDrive/CS182_masking_harness/runs`. The refreshed corpus cache ZIP is saved there after tokenizing.

Set `WORKFLOW = 'train'` to train or resume, or `restore` to view saved results. This notebook does not extract ZIP archives.

## 1 · Shared corpus

Train on the `train` split and evaluate on the `test` split. `DATA_ID` and `DATA_REV` pin the dataset snapshot; `MODEL_CONFIG_ID` and `MODEL_CONFIG_REV` fetch only the tokenizer and architecture config. Training starts from fresh SimpleStories-initialized weights.

In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import itertools
from zipfile import ZIP_STORED, ZipFile
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CS182_masking_harness')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

CANDIDATES = [Path('/content/harness'), Path.cwd() / 'harness', Path.cwd()]
HARNESS_DIR = next((p for p in CANDIDATES if (p / 'data.py').exists()), None)
if HARNESS_DIR is None:
    raise FileNotFoundError('Make the harness folder available at /content/harness first.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(HARNESS_DIR / 'requirements.txt')])
sys.path.insert(0, str(HARNESS_DIR))

import numpy as np
import pandas as pd
import torch
from transformers import AutoConfig, AutoTokenizer
from data import (DATA_ID, DATA_REV, DEFAULT_SHUFFLE_SEED,
                  MODEL_CONFIG_ID, MODEL_CONFIG_REV,
                  load_or_prepare_corpus)
from training_fcm import (DEFAULT_BATCH_SIZE, DEFAULT_CHECKPOINT_EVERY,
                           DEFAULT_LEARNING_RATE, DEFAULT_SEED, DEFAULT_SEQUENCE_LENGTH,
                           DEFAULT_TEST_BLOCKS, DEFAULT_WARMUP_STEPS, DEFAULT_WEIGHT_DECAY,
                           TrainConfig, profile_peak_memory, recommend_workers,
                           run_directory, run_many)
from masking import DEFAULT_BASE_MASK_RATE, DEFAULT_CONTENT_WEIGHT, DEFAULT_OTHER_WEIGHT
from analysis import plot_test_curves, summarize_test

ROOT = DRIVE_ROOT
RUN_ROOT = ROOT / 'runs'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = HARNESS_DIR / 'cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Select a Colab GPU runtime before running this notebook.')
print(f'GPU: {torch.cuda.get_device_name(DEVICE)}')


In [ ]:
# Reuse files already present in this runtime; this notebook never extracts archives.
WORKFLOW = 'train'  # 'restore' or 'train'
if WORKFLOW not in {'restore', 'train'}:
    raise ValueError("WORKFLOW must be 'restore' or 'train'")

RESTORE_ONLY = WORKFLOW == 'restore'
cache_complete = any(CACHE_ROOT.glob('corpus-*/complete.json'))
LOAD_TRAIN_CORPUS = WORKFLOW == 'train'
print(f'Restore-only mode: {RESTORE_ONLY}; cache files found: {cache_complete}')
if cache_complete:
    print(f'Corpus cache: {CACHE_ROOT}')
if WORKFLOW == 'train':
    print('A compatible train/test cache is reused; otherwise it is rebuilt.')
if RESTORE_ONLY and not cache_complete:
    print('No corpus cache found; training data will not be loaded.')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG_ID, revision=MODEL_CONFIG_REV, use_fast=True)
model_config = AutoConfig.from_pretrained(MODEL_CONFIG_ID, revision=MODEL_CONFIG_REV)
model_config.use_cache = False
assert tokenizer.is_fast and tokenizer.eos_token_id is not None
assert len(tokenizer) <= model_config.vocab_size

PRESET = 'data_constrained'  # 'data_constrained', 'full_budget', or 'smoke'
REGIMES = ('clean', 'uniform', 'uniform_mrd')  # Add 'pos' or 'pos_mrd' to include POS weighting
SEEDS = (DEFAULT_SEED, DEFAULT_SEED + 1, DEFAULT_SEED + 2)
MAX_CONCURRENT_RUNS = 6  # VRAM profiling may recommend fewer; this is the upper bound.
COMMON_TRAINING = dict(test_blocks=DEFAULT_TEST_BLOCKS, batch_size=DEFAULT_BATCH_SIZE,
                       micro_batch_size=32, sequence_length=DEFAULT_SEQUENCE_LENGTH,
                       warmup_steps=DEFAULT_WARMUP_STEPS)

if PRESET == 'data_constrained':
    base = {**COMMON_TRAINING, 'train_targets': 100_663_296}
    # 96, 24, and 6 Mi unique tokens yield 1, 4, and 16 passes respectively.
    unique_budgets = [100_663_296, 25_165_824, 6_291_456]
    RUN_PLAN = [
        {'name': f'{regime}-data{budget // 1_048_576}Mi-seed{DEFAULT_SEED}',
         'seed': DEFAULT_SEED, 'regime': regime, 'unique_train_tokens': budget}
        for regime, budget in itertools.product(REGIMES, unique_budgets)
    ]
elif PRESET == 'full_budget':
    base = {**COMMON_TRAINING, 'train_targets': 110_000_000}
    RUN_PLAN = [
        {'name': f'{regime}-seed{seed}', 'seed': seed, 'regime': regime}
        for regime, seed in itertools.product(REGIMES, SEEDS)
    ]
elif PRESET == 'smoke':
    base = {**COMMON_TRAINING, 'train_targets': 2_048, 'test_blocks': 8,
            'batch_size': 2, 'micro_batch_size': 2, 'sequence_length': 64,
            'warmup_steps': 4}
    RUN_PLAN = [
        {'name': f'{regime}-smoke', 'seed': DEFAULT_SEED, 'regime': regime}
        for regime in REGIMES
    ]
else:
    raise ValueError("PRESET must be 'data_constrained', 'full_budget', or 'smoke'")

base.update(learning_rate=DEFAULT_LEARNING_RATE,
            weight_decay=DEFAULT_WEIGHT_DECAY,
            base_mask_rate=DEFAULT_BASE_MASK_RATE,
            content_weight=DEFAULT_CONTENT_WEIGHT, other_weight=DEFAULT_OTHER_WEIGHT,
            checkpoint_every=DEFAULT_CHECKPOINT_EVERY)
configs = [TrainConfig(**{**base, **entry}) for entry in RUN_PLAN]
assert configs and len({cfg.name for cfg in configs}) == len(configs)
assert all(cfg.sequence_length <= model_config.max_position_embeddings for cfg in configs)
assert all(cfg.batch_size % cfg.micro_batch_size == 0 for cfg in configs)
train_tokens = max(cfg.unique_train_blocks * cfg.sequence_length for cfg in configs)
test_tokens = max(cfg.test_blocks * cfg.sequence_length for cfg in configs)
print(f'{len(configs)} runs: {", ".join(REGIMES)}; unique-data budgets up to {train_tokens:,} tokens')


In [ ]:
if LOAD_TRAIN_CORPUS:
    cache, DATA_SPEC, train_ids, test_ids, train_pos = load_or_prepare_corpus(
        tokenizer, CACHE_ROOT, train_tokens, test_tokens,
        shuffle_seed=DEFAULT_SHUFFLE_SEED,
        include_pos=True,
    )
    assert len(train_ids) == train_tokens + 1
    assert len(test_ids) == test_tokens + 1
    assert len(train_pos) == train_tokens + 1
    print(f'Loaded train: {train_tokens:,} tokens; test: {test_tokens:,}')
    corpus_bundle = DRIVE_ROOT / 'corpus_cache.zip'
    temporary_bundle = corpus_bundle.with_name(corpus_bundle.name + '.tmp')
    with ZipFile(temporary_bundle, 'w', compression=ZIP_STORED) as archive:
        for path in sorted(cache.iterdir()):
            if path.is_file():
                archive.write(path, Path('cache') / cache.name / path.name)
    temporary_bundle.replace(corpus_bundle)
    print(f'Saved cache ZIP to {corpus_bundle}')
else:
    cache = None
    DATA_SPEC = None
    train_ids = test_ids = train_pos = None
    print('Skipping corpus loading in restore-only mode.')

history_path = RUN_ROOT / 'test_history.csv'
history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
if history.empty and RESTORE_ONLY:
    saved_rows = []
    for cfg in configs:
        run_dir = run_directory(cfg, RUN_ROOT)
        metrics_file = run_dir / 'metrics.json'
        checkpoints = sorted(run_dir.glob('step-*.pt'))
        if metrics_file.exists():
            saved_rows.extend(json.loads(metrics_file.read_text()))
        elif checkpoints:
            state = torch.load(checkpoints[-1], map_location='cpu', weights_only=True)
            saved_rows.extend(state.get('history', []))
    if saved_rows:
        history = pd.DataFrame(saved_rows)
if not history.empty:
    print(f'Loaded {len(history)} saved test rows')


## 2 · Masking conditions

Clean uses ordinary causal attention. Uniform FCM hides eligible whole words at a fixed 15% rate. Uniform + MRD decays that rate linearly from 30% to zero over training. The selected positions are hidden as attention keys, while all original next-token targets remain. The first position of each packed block is protected. Fixed-rate POS-weighted FCM (`pos`) and POS-weighted MRD (`pos_mrd`) remain available: add either name to `REGIMES` in the configuration cell.

## 3 · Train

The focused plan crosses three masking conditions with 96, 24, and 6 Mi unique-token budgets: nine runs total. Total target exposure is fixed across the budgets. Effective batch size is 128, using micro-batches of 32. The trainer profiles one step and chooses the concurrent-run count from measured VRAM, up to `MAX_CONCURRENT_RUNS`.

In [ ]:
if RESTORE_ONLY:
    print("Restore-only mode: skipping training. Set WORKFLOW to train above to intentionally start or resume.")
else:
    profile_config = next((c for c in configs if c.regime != 'clean'), configs[0])
    profile = profile_peak_memory(profile_config, train_ids, train_pos)
    reserve_gib = min(5.0, max(2.5, profile['total_bytes'] / 1024**3 * 0.20))
    MAX_PARALLEL = recommend_workers(profile, reserve_gib=reserve_gib,
                                     max_workers=MAX_CONCURRENT_RUNS)
    print(f'GPU memory reserve: {reserve_gib:.1f} GiB (20% of VRAM, bounded 2.5–5 GiB)')
    print(json.dumps({**profile, 'recommended_concurrent_runs': MAX_PARALLEL}, indent=2))

    run_records = run_many(configs, train_ids, test_ids, train_pos, RUN_ROOT,
                           max_workers=MAX_PARALLEL)
    records = [row for run in run_records for row in run]
    history = pd.DataFrame(records).sort_values(['name', 'step'])
    history.to_csv(RUN_ROOT / 'test_history.csv', index=False)
    print(f'Finished {len(configs)} runs. Results: {RUN_ROOT}')


## 4 · Test results

Training uses the train split, and the test split is evaluated at each checkpoint. Checkpoints and reports are written directly to Drive. The cache ZIP is refreshed in Drive immediately after tokenizing finishes.

In [ ]:
if history.empty:
    print('No test history is available yet; train or restore completed runs first.')
else:
    ranking = summarize_test(history, RUN_ROOT)
    display(ranking.round(5))
    test_figures = plot_test_curves(history, RUN_ROOT)
    for figure in test_figures:
        display(figure)


In [ ]:
# Compare each configured regime at the smallest configured data budget and default seed.
from sample_generations import DEFAULT_PROMPT, generate_runs

generation_configs = [cfg for cfg in configs if cfg.seed == DEFAULT_SEED]
if not generation_configs:
    raise ValueError(f'No run uses DEFAULT_SEED={DEFAULT_SEED}')
generation_budget = min(cfg.unique_train_blocks for cfg in generation_configs)
generation_configs = [cfg for cfg in generation_configs
                      if cfg.unique_train_blocks == generation_budget]
run_dirs = [run_directory(cfg, RUN_ROOT) for cfg in generation_configs]

print(f'Prompt: {DEFAULT_PROMPT}')
print(f'Comparing {len(run_dirs)} runs at {generation_budget * configs[0].sequence_length:,} unique tokens')
generate_runs(run_dirs, prompt=DEFAULT_PROMPT, max_new_tokens=100,
              temperature=0.8, top_p=0.9, seed=DEFAULT_SEED)
